# TCGA (BRCA, LIHC, COAD): wsRGES with corrected dose/time offsets, cell-line weights per configuration and the CS p-value filter

Per-drug scores of the three TCGA cancers of the Chen 2017 ChEMBL IC50 validation, from the per-instance connectivity scores (CS) of
the 71 LINCS lines (`src/validations/chembl/wsRGES_tcga_corrected_offsets.py`). This notebook summarises its outputs: the
reproduction of the stored `binchen_r` table, the effect of the 64-line weights on the MITHrIL configurations, the effect of the offset assignment
(`binchen_r` vs `corrected`), the CS p-value filter, and the two offset variants after the filter. IC50 correlations are not computed here.
Sign convention: more negative = stronger predicted reversal of the disease signature.

## Definitions

| Term | What it is | Where computed |
|---|---|---|
| **Configuration** (`config`) | Representation of the disease and drug signatures: `DEG`, `mith`, `mith_LM`, `mith_pw`. | CS runs of `tcga_wsRGES_cs_runs_used.tsv` |
| **Collapse** | Per-drug summary of the instance CS (drug = `pert_iname`): `wsRGES` (weighted by the TCGA weights, all weighted lines), `wsRGES_chen2017w` (Chen 2017 weights), `sRGES_all` (all 71 lines, unweighted), `best_all` (most negative CS, all lines), `srges` and `best` (validation line only). | `wsRGES_tcga_corrected_offsets.collapse` |
| **Validation line** | MCF7 (BRCA), HEPG2 (LIHC), HT29 (COAD). | `wsRGES_IC50_correlations.CANCERS` |
| **Dose/time offset** | Mean CS(10 µM, 24 h) - CS(other instance) over pairs of the same drug and cell line, per dose bin (low < 10 µM, high) and time (6, 24 h) of the other instance; added to each instance CS before the sRGES mean. `best` and `best_all` use no offset. | `sRGES.build_reference_diff` |
| **`binchen_r`** | Offset-to-instance mapping of the published R code: low dose / 24 h receives the high 6 h offset, high dose / 6 h the high 24 h offset. | `sRGES._get_sRGES_one` |
| **`corrected`** | Each instance receives the offset of its own dose bin and time. | same |
| **Weights, 58 / 64 lines** | 58: TCGA tumour correlation of the DepMap profiles (`tcga_cell_line_weights_ccle58.tsv`), the Chen 2017 line set, used by `DEG` (Chen 2017 reproduction, `tcga_wsRGES_per_drug_DEG__binchen_r.tsv`); 64: plus six lines weighted from HPA / GEO profiles (`tcga_cell_line_weights.tsv`, `report_tcga_cell_line_weights.md`), used by `mith`, `mith_LM`, `mith_pw`. One weight set per configuration, in every offset assignment and filter variant. | `tcga_ccle_cell_line_weights.py` |
| **p filter (`bh05`)** | p' = (k + 1) / 1001 with k = `cs_p_value` x 1000, Benjamini-Hochberg within one CS run (one cancer x configuration x cell line), instances with p_adj <= 0.05 kept. | `case_study.cs_pvalue_filter.flag` |
| **Rebuilt / frozen offsets** | After the filter: `bh05` rebuilds the offset table from the kept instances; `bh05f` scores them with the table of all unfiltered instances. | `wsRGES_tcga_corrected_offsets.py` |
| **Percentile** | Rank of the score within (cancer, configuration, collapse) / number of drugs x 100; low = strong reversal. | this notebook |


In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display, Markdown, Image

ROOT = Path(os.getcwd()).resolve()
while not (ROOT / 'connectivity_score' / 'input').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import conf
import lfs_style
lfs_style.apply()
OUT = conf.BINCHEN_VALIDATION_OUT_DIR
IMG = conf.IMG_DIR / 'binchen2017_validation'
REL = '../imgs/binchen2017_validation'
CANCERS = ['BRCA', 'LIHC', 'COAD']
CFG = ['DEG', 'mith', 'mith_LM', 'mith_pw']
SR = ['wsRGES', 'wsRGES_chen2017w', 'sRGES_all', 'srges']
K = ['cancer', 'config', 'collapse', 'pert_iname']
G = ['cancer', 'config', 'collapse']

def md(df, index=False, fmt='{:.3f}'):
    df = df.reset_index() if index else df
    def f(v):
        if isinstance(v, (float, np.floating)):
            return '' if np.isnan(v) else (fmt.format(v) if not float(v).is_integer() or abs(v) < 1 else f'{v:.0f}')
        return str(v)
    lines = ['| ' + ' | '.join(map(str, df.columns)) + ' |', '|' + '---|' * len(df.columns)]
    lines += ['| ' + ' | '.join(f(v) for v in r) + ' |' for r in df.itertuples(index=False)]
    display(Markdown(chr(10).join(lines)))

def rd(n):
    return pd.read_csv(OUT / n, sep='\t')
summ = rd('tcga_wsRGES_corrected_offsets_summary.tsv')
stored = rd('tcga_wsRGES_per_drug.tsv')
cor64, cor58 = rd('tcga_wsRGES_per_drug_corrected.tsv'), rd('tcga_wsRGES_per_drug_corrected_ccle58.tsv')
f64, ff64 = rd('tcga_wsRGES_per_drug_corrected_pvalue_filtered.tsv'), rd('tcga_wsRGES_per_drug_corrected_pvalue_filtered_frozen_offsets.tsv')
display(Markdown(f"Rows: stored `binchen_r` {len(stored):,}; `corrected` (weights per configuration) {len(cor64):,}; `corrected` 58 lines {len(cor58):,}; "
                 f"filtered rebuilt {len(f64):,}, frozen {len(ff64):,}."))

Rows: stored `binchen_r` 657,528; `corrected` (weights per configuration) 659,805; `corrected` 58 lines 657,528; filtered rebuilt 362,810, frozen 362,810.

## 1. Reproduction of the stored table

The script recomputes every collapse with `binchen_r` and the 58 DepMap weights, the settings of the stored `tcga_wsRGES_per_drug.tsv`,
and compares the two tables drug by drug. This is a regression test: agreement means that the instance loading, the metadata merge, the weights and the
collapse code give the same result as when the stored table was written, so the other tables differ from it only by what is varied on purpose.
The agreement of `binchen_r` with the published R code is tested separately (`src/validations/binchen2017/test_sRGES_unit.py`, `test_sRGES_replication.py`).

In [2]:
t = summ.groupby('collapse').agg(rows=('cancer', 'size'), max_abs_diff=('max_abs_diff_binchen_r_vs_stored', 'max'),
                                  drugs_in_one_table_only=('n_drugs_only_one_table', 'sum')).reset_index()
display(Markdown("**Table 1.** Recomputed `binchen_r` (58 lines) vs stored table, per collapse, over the 12 cancer x configuration rows "
                 "(`tcga_wsRGES_corrected_offsets_summary.tsv`). `max_abs_diff`: largest absolute score difference (CS units); "
                 "`drugs_in_one_table_only`: drugs present in only one of the two tables (count)."))
t['max_abs_diff'] = t['max_abs_diff'].map(lambda v: f'{v:.1e}')
md(t)

**Table 1.** Recomputed `binchen_r` (58 lines) vs stored table, per collapse, over the 12 cancer x configuration rows (`tcga_wsRGES_corrected_offsets_summary.tsv`). `max_abs_diff`: largest absolute score difference (CS units); `drugs_in_one_table_only`: drugs present in only one of the two tables (count).

| collapse | rows | max_abs_diff | drugs_in_one_table_only |
|---|---|---|---|
| best | 12 | 0.0e+00 | 0 |
| best_all | 12 | 0.0e+00 | 0 |
| sRGES_all | 12 | 1.1e-16 | 0 |
| srges | 12 | 1.0e-16 | 0 |
| wsRGES | 12 | 1.0e-16 | 0 |
| wsRGES_chen2017w | 12 | 1.0e-16 | 0 |

## 2. Effect of the 64-line weights on the MITHrIL configurations (corrected offsets)

`DEG` keeps the 58 DepMap lines of Chen 2017, so its scores equal those of the 58-line table (asserted below).

Only the weighted collapses (`wsRGES`) use the TCGA weights; `wsRGES_chen2017w` uses the Chen 2017 weights, which have no
entry for the six lines. The largest weight is unchanged in every cancer (it belongs to a DepMap line), so the instances of the 58
DepMap lines keep their weighted score; the 64-line wsRGES adds the instances of the six lines.

In [3]:
j = cor58.merge(cor64, on=K, how='outer', suffixes=('_58', '_64'), indicator=True)
rows = []
for (c, g, k), d in j.groupby(G, sort=False):
    b = d[d['_merge'] == 'both']
    pa = b['score_58'].rank(pct=True) * 100; pb = b['score_64'].rank(pct=True) * 100
    rows.append({'cancer': c, 'config': g, 'collapse': k, 'drugs_58': int(d['score_58'].notna().sum()), 'drugs_64': int(d['score_64'].notna().sum()),
                 'only_64': int((d['_merge'] == 'right_only').sum()), 'changed': int((b['score_58'] - b['score_64']).abs().gt(1e-12).sum()),
                 'spearman_shared': spearmanr(b['score_58'], b['score_64']).statistic,
                 'median_abs_dpct': (pa - pb).abs().median(), 'max_abs_dpct': (pa - pb).abs().max()})
w = pd.DataFrame(rows)
assert (w[w.collapse != 'wsRGES']['changed'] == 0).all() and (w[w.collapse != 'wsRGES']['only_64'] == 0).all()
assert (w[w.config == 'DEG'][['changed', 'only_64']] == 0).all().all()
display(Markdown("**Table 2.** `wsRGES` with corrected offsets, 58-line vs 64-line weights, per cancer and MITHrIL configuration. `drugs_58`, `drugs_64`: "
                 "drugs scored (count); `only_64`: drugs scored only with the 64-line weights; `changed`: drugs in both whose score differs "
                 "(> 1e-12); `spearman_shared`: Spearman of the two scores on the drugs in both; `median_abs_dpct`, `max_abs_dpct`: absolute change of "
                 "percentile (0-100) on the drugs in both. Every other collapse is identical in the two tables."))
md(w[(w.collapse == 'wsRGES') & (w.config != 'DEG')].drop(columns='collapse'))

**Table 2.** `wsRGES` with corrected offsets, 58-line vs 64-line weights, per cancer and MITHrIL configuration. `drugs_58`, `drugs_64`: drugs scored (count); `only_64`: drugs scored only with the 64-line weights; `changed`: drugs in both whose score differs (> 1e-12); `spearman_shared`: Spearman of the two scores on the drugs in both; `median_abs_dpct`, `max_abs_dpct`: absolute change of percentile (0-100) on the drugs in both. Every other collapse is identical in the two tables.

| cancer | config | drugs_58 | drugs_64 | only_64 | changed | spearman_shared | median_abs_dpct | max_abs_dpct |
|---|---|---|---|---|---|---|---|---|
| BRCA | mith | 12168 | 12421 | 253 | 1581 | 0.996 | 0.345 | 48.652 |
| BRCA | mith_LM | 12168 | 12421 | 253 | 1581 | 0.995 | 0.370 | 61.177 |
| BRCA | mith_pw | 12168 | 12421 | 253 | 1581 | 0.996 | 0.288 | 51.290 |
| COAD | mith | 12168 | 12421 | 253 | 1581 | 0.996 | 0.468 | 38.774 |
| COAD | mith_LM | 12168 | 12421 | 253 | 1581 | 0.996 | 0.444 | 45.176 |
| COAD | mith_pw | 12168 | 12421 | 253 | 1581 | 0.997 | 0.468 | 48.932 |
| LIHC | mith | 12168 | 12421 | 253 | 1581 | 0.996 | 0.329 | 42.521 |
| LIHC | mith_LM | 12168 | 12421 | 253 | 1581 | 0.995 | 0.337 | 55.630 |
| LIHC | mith_pw | 12168 | 12421 | 253 | 1581 | 0.996 | 0.485 | 50.115 |

## 3. Effect of the offset assignment: `binchen_r` vs `corrected`

In [4]:
s = summ[summ.collapse.isin(SR)].pivot_table(index=['cancer', 'config'], columns='collapse', values='spearman_binchen_r_vs_corrected')[SR]
s = s.reindex(pd.MultiIndex.from_product([CANCERS, CFG], names=['cancer', 'config']))
display(Markdown("**Table 3.** Spearman correlation of the `binchen_r` and `corrected` scores on the shared drugs, both with the weight set "
                 "of the configuration (58 lines for `DEG`, 64 for the MITHrIL configurations), per cancer, configuration and offset-using collapse (`spearman_binchen_r_vs_corrected` of the summary table). "
                 "`best` and `best_all` use no offset and are identical in the two assignments."))
md(s, index=True)

**Table 3.** Spearman correlation of the `binchen_r` and `corrected` scores on the shared drugs, both with the weight set of the configuration (58 lines for `DEG`, 64 for the MITHrIL configurations), per cancer, configuration and offset-using collapse (`spearman_binchen_r_vs_corrected` of the summary table). `best` and `best_all` use no offset and are identical in the two assignments.

| cancer | config | wsRGES | wsRGES_chen2017w | sRGES_all | srges |
|---|---|---|---|---|---|
| BRCA | DEG | 0.958 | 0.957 | 0.962 | 0.974 |
| BRCA | mith | 0.985 | 0.983 | 0.985 | 0.973 |
| BRCA | mith_LM | 0.996 | 0.996 | 0.997 | 0.992 |
| BRCA | mith_pw | 0.992 | 0.992 | 0.992 | 0.985 |
| LIHC | DEG | 0.992 | 0.992 | 0.993 | 0.999 |
| LIHC | mith | 0.966 | 0.963 | 0.968 | 0.995 |
| LIHC | mith_LM | 0.988 | 0.987 | 0.990 | 0.998 |
| LIHC | mith_pw | 1.000 | 1.000 | 1.000 | 1.000 |
| COAD | DEG | 0.992 | 0.992 | 0.993 | 0.992 |
| COAD | mith | 0.997 | 0.997 | 0.997 | 0.986 |
| COAD | mith_LM | 0.994 | 0.994 | 0.994 | 0.997 |
| COAD | mith_pw | 0.999 | 0.999 | 0.999 | 0.968 |

In [5]:
x = stored.merge(cor58, on=K, suffixes=('_r', '_c'))
assert (x[x.collapse.isin(['best', 'best_all'])].eval('score_r - score_c').abs().max()) < 1e-12
x = x[x.collapse.isin(SR)]
for v in ('r', 'c'):
    x['pct_' + v] = x.groupby(G)['score_' + v].rank(pct=True) * 100
x['adp'] = (x['pct_c'] - x['pct_r']).abs()
t = x.groupby(G)['adp'].agg(lambda a: f'{a.median():.1f} / {a.quantile(.95):.1f} / {a.max():.1f}').unstack('collapse')[SR]
t = t.reindex(pd.MultiIndex.from_product([CANCERS, CFG], names=['cancer', 'config']))
display(Markdown("**Table 4.** Absolute change of percentile (0-100) from `binchen_r` to `corrected`, median / 95th percentile / maximum over the "
                 "drugs of each (cancer, configuration, collapse). Computed on the 58-line weights (stored table vs "
                 "`tcga_wsRGES_per_drug_corrected_ccle58.tsv`), the only pair of saved tables that differ by the offset assignment alone."))
md(t, index=True)

**Table 4.** Absolute change of percentile (0-100) from `binchen_r` to `corrected`, median / 95th percentile / maximum over the drugs of each (cancer, configuration, collapse). Computed on the 58-line weights (stored table vs `tcga_wsRGES_per_drug_corrected_ccle58.tsv`), the only pair of saved tables that differ by the offset assignment alone.

| cancer | config | wsRGES | wsRGES_chen2017w | sRGES_all | srges |
|---|---|---|---|---|---|
| BRCA | DEG | 6.7 / 15.8 / 25.1 | 6.6 / 16.2 / 26.5 | 6.1 / 15.0 / 21.9 | 4.6 / 13.7 / 16.9 |
| BRCA | mith | 4.0 / 9.0 / 13.8 | 4.2 / 9.5 / 15.6 | 3.7 / 9.4 / 10.3 | 3.4 / 16.2 / 27.4 |
| BRCA | mith_LM | 1.5 / 5.2 / 7.6 | 1.4 / 5.4 / 8.4 | 1.7 / 4.8 / 7.4 | 2.2 / 7.3 / 8.3 |
| BRCA | mith_pw | 2.5 / 6.9 / 10.1 | 2.6 / 7.1 / 10.1 | 2.6 / 7.3 / 9.3 | 2.7 / 12.0 / 18.1 |
| LIHC | DEG | 1.5 / 7.6 / 17.3 | 1.6 / 7.7 / 17.0 | 1.1 / 7.4 / 10.3 | 0.2 / 1.8 / 13.2 |
| LIHC | mith | 6.4 / 13.2 / 19.8 | 6.7 / 13.7 / 19.7 | 6.2 / 12.6 / 16.8 | 0.6 / 1.5 / 28.1 |
| LIHC | mith_LM | 3.5 / 9.0 / 19.5 | 3.5 / 9.2 / 19.9 | 3.1 / 8.5 / 12.0 | 0.3 / 3.2 / 27.3 |
| LIHC | mith_pw | 0.2 / 1.1 / 2.2 | 0.3 / 1.1 / 2.3 | 0.2 / 1.0 / 1.4 | 0.1 / 0.4 / 4.8 |
| COAD | DEG | 2.0 / 7.8 / 15.2 | 2.1 / 7.7 / 15.8 | 1.6 / 7.8 / 12.2 | 2.7 / 7.1 / 11.1 |
| COAD | mith | 1.2 / 4.8 / 7.3 | 1.2 / 4.9 / 7.6 | 1.2 / 4.9 / 7.0 | 4.6 / 7.2 / 7.9 |
| COAD | mith_LM | 2.2 / 5.9 / 11.9 | 2.3 / 6.2 / 12.9 | 2.2 / 5.7 / 7.2 | 1.8 / 4.7 / 7.7 |
| COAD | mith_pw | 1.1 / 2.4 / 4.8 | 1.2 / 2.4 / 4.6 | 1.1 / 2.3 / 3.8 | 6.2 / 12.8 / 13.5 |

In [6]:
top = x.assign(d=x['pct_c'] - x['pct_r']).sort_values('adp', ascending=False)
top = top[top.collapse == 'wsRGES'].groupby(['cancer', 'config']).head(3)
display(Markdown("**Table 5.** The three drugs with the largest percentile change per cancer and configuration, `wsRGES` (58 lines). "
                 "`pct_r`, `pct_c`: percentile with `binchen_r` and `corrected` (0-100, low = strong reversal); `n_instances`: instances averaged. "
                 f"All have {int(top['n_instances_r'].min())}-{int(top['n_instances_r'].max())} instances: with few instances the offset of a single instance sets the score."))
md(top[['cancer', 'config', 'pert_iname', 'n_instances_r', 'pct_r', 'pct_c']].rename(columns={'n_instances_r': 'n_instances'}), fmt='{:.1f}')

**Table 5.** The three drugs with the largest percentile change per cancer and configuration, `wsRGES` (58 lines). `pct_r`, `pct_c`: percentile with `binchen_r` and `corrected` (0-100, low = strong reversal); `n_instances`: instances averaged. All have 1-3 instances: with few instances the offset of a single instance sets the score.

| cancer | config | pert_iname | n_instances | pct_r | pct_c |
|---|---|---|---|---|---|
| BRCA | DEG | BRD-A32440707 | 1 | 58.9 | 33.8 |
| BRCA | DEG | pempidine | 1 | 63.1 | 38.0 |
| BRCA | DEG | BD-1008 | 1 | 58.1 | 33.0 |
| LIHC | mith | tolmetin | 1 | 6.7 | 26.5 |
| LIHC | mith | BRD-K49477212 | 1 | 5.4 | 25.1 |
| LIHC | mith_LM | BRD-K51731619 | 1 | 31.7 | 12.2 |
| LIHC | mith_LM | BRD-A17664363 | 1 | 28.0 | 8.8 |
| LIHC | mith | stiripentol | 1 | 2.9 | 21.5 |
| LIHC | DEG | BRD-K81538844 | 1 | 54.9 | 37.6 |
| LIHC | DEG | 3,6-dimethoxyflavone | 1 | 51.8 | 35.0 |
| LIHC | DEG | BRD-K73334721 | 1 | 50.1 | 33.7 |
| LIHC | mith_LM | BRD-K81538844 | 1 | 42.4 | 26.2 |
| COAD | DEG | BRD-K29500995 | 2 | 56.8 | 41.5 |
| COAD | DEG | nifekalant | 2 | 62.0 | 46.7 |
| COAD | DEG | dimercaptosuccinic-acid | 2 | 67.7 | 52.7 |
| BRCA | mith | BRD-K27682288 | 1 | 34.1 | 20.3 |
| BRCA | mith | BRD-K96178772 | 1 | 34.4 | 20.7 |
| BRCA | mith | BRD-K09023525 | 1 | 33.3 | 19.6 |
| COAD | mith_LM | BRD-K80726792 | 1 | 27.3 | 15.4 |
| COAD | mith_LM | BRD-K89288521 | 1 | 25.1 | 13.3 |
| COAD | mith_LM | BRD-K97748079 | 1 | 21.3 | 10.9 |
| BRCA | mith_pw | SA-102676 | 2 | 58.3 | 48.2 |
| BRCA | mith_pw | BRD-K41520872 | 1 | 63.7 | 53.8 |
| BRCA | mith_pw | BRD-K88626416 | 1 | 64.3 | 54.5 |
| BRCA | mith_LM | pazufloxacin | 1 | 18.7 | 26.3 |
| BRCA | mith_LM | BRD-K61556970 | 1 | 23.7 | 31.3 |
| BRCA | mith_LM | BRD-K70068422 | 1 | 18.7 | 26.3 |
| COAD | mith | BRD-K62917576 | 1 | 88.6 | 81.3 |
| COAD | mith | BRD-K37140349 | 1 | 88.9 | 81.6 |
| COAD | mith | BRD-K30855709 | 1 | 91.3 | 84.0 |
| COAD | mith_pw | BRD-K96539190 | 3 | 52.0 | 47.2 |
| COAD | mith_pw | BRD-K48752687 | 1 | 60.3 | 55.7 |
| COAD | mith_pw | BRD-K93800099 | 1 | 82.4 | 77.9 |
| LIHC | mith_pw | anastrozole | 1 | 85.0 | 82.8 |
| LIHC | mith_pw | BRD-A17664363 | 1 | 78.9 | 76.7 |
| LIHC | mith_pw | thenoyltrifluoroacetone | 1 | 34.5 | 32.4 |

## 4. The CS p-value filter

The filter acts only where the Monte Carlo p-values are spread. Table 6 gives the p-values of 4 CS runs (cell lines) drawn at random
(seed 0) per cancer and configuration.

In [7]:
from loader import load_cs_run
u = pd.read_csv(OUT / 'tcga_wsRGES_cs_runs_used.tsv', sep='\t')
rows = []
for c in CANCERS:
    for g in CFG:
        for _, r in u[(u.cancer == c) & (u.config == g)].sample(4, random_state=0).iterrows():
            p = load_cs_run(disease=c, landmark_disease=False, selected_cs_run_id=r.cs_run_id, verbose=False)[0]['cs_p_value']
            rows.append({'cancer': c, 'config': g, 'frac_p0': (p == 0).mean(), 'frac_p_le_0.05': (p <= .05).mean()})
q = pd.DataFrame(rows).groupby(['cancer', 'config'], sort=False).agg(['min', 'max'])
q.columns = [f'{a} {b}' for a, b in q.columns]
display(Markdown("**Table 6.** Monte Carlo p-values (`cs_p_value`, 1,000 random gene sets) of 4 randomly drawn CS runs per cancer and configuration: "
                 "minimum and maximum over the 4 runs of the fraction of instances with p = 0 and with p <= 0.05 (fraction 0-1). "
                 "With p = 0 for every instance, the BH filter keeps every instance and its result carries no information on the scores."))
md(q, index=True)

**Table 6.** Monte Carlo p-values (`cs_p_value`, 1,000 random gene sets) of 4 randomly drawn CS runs per cancer and configuration: minimum and maximum over the 4 runs of the fraction of instances with p = 0 and with p <= 0.05 (fraction 0-1). With p = 0 for every instance, the BH filter keeps every instance and its result carries no information on the scores.

| cancer | config | frac_p0 min | frac_p0 max | frac_p_le_0.05 min | frac_p_le_0.05 max |
|---|---|---|---|---|---|
| BRCA | DEG | 0.013 | 0.200 | 0.261 | 0.500 |
| BRCA | mith | 1 | 1 | 1 | 1 |
| BRCA | mith_LM | 0.005 | 0.100 | 0.118 | 0.458 |
| BRCA | mith_pw | 0.820 | 1 | 0.970 | 1 |
| LIHC | DEG | 0.040 | 0.231 | 0.300 | 0.548 |
| LIHC | mith | 1 | 1 | 1 | 1 |
| LIHC | mith_LM | 0.000 | 0.005 | 0.090 | 0.142 |
| LIHC | mith_pw | 0.805 | 1 | 0.967 | 1 |
| COAD | DEG | 0.000 | 0.010 | 0.058 | 0.150 |
| COAD | mith | 1 | 1 | 1 | 1 |
| COAD | mith_LM | 0.000 | 0.000 | 0.000 | 0.073 |
| COAD | mith_pw | 0.790 | 1 | 0.970 | 1 |

In [8]:
a = summ[summ.collapse == 'wsRGES'][['cancer', 'config', 'n_instances_before', 'n_instances_kept']].copy()
a['frac_kept'] = a['n_instances_kept'] / a['n_instances_before']
d = summ.pivot_table(index=['cancer', 'config'], columns='collapse', values='n_drugs_after_filter')
b = summ.pivot_table(index=['cancer', 'config'], columns='collapse', values='n_drugs_before')
t = a.set_index(['cancer', 'config'])
for k in ['wsRGES', 'sRGES_all', 'srges', 'best']:
    t[k] = [f'{int(d.loc[i, k])} / {int(b.loc[i, k])}' if i in d.index and not np.isnan(d.loc[i, k]) else f'0 / {int(b.loc[i, k])}' for i in t.index]
t = t.reindex(pd.MultiIndex.from_product([CANCERS, CFG], names=['cancer', 'config']))
display(Markdown("**Table 7.** Instances and drugs before and after the BH filter, per cancer and configuration. `n_instances_before`, "
                 "`n_instances_kept`: instances (count, 71 lines); `frac_kept`: fraction kept (0-1); collapse columns: drugs scored after / before "
                 "the filter (count); `srges` and `best` are the validation-line collapses, the drugs whose IC50 can be correlated on that line."))
md(t, index=True)

**Table 7.** Instances and drugs before and after the BH filter, per cancer and configuration. `n_instances_before`, `n_instances_kept`: instances (count, 71 lines); `frac_kept`: fraction kept (0-1); collapse columns: drugs scored after / before the filter (count); `srges` and `best` are the validation-line collapses, the drugs whose IC50 can be correlated on that line.

| cancer | config | n_instances_before | n_instances_kept | frac_kept | wsRGES | sRGES_all | srges | best |
|---|---|---|---|---|---|---|---|---|
| BRCA | DEG | 66511 | 7035 | 0.106 | 2257 / 12168 | 2467 / 12442 | 1036 / 3982 | 1036 / 3982 |
| BRCA | mith | 66511 | 66469 | 0.999 | 12417 / 12421 | 12438 / 12442 | 3981 / 3982 | 3981 / 3982 |
| BRCA | mith_LM | 66511 | 3094 | 0.047 | 1422 / 12421 | 1441 / 12442 | 251 / 3982 | 251 / 3982 |
| BRCA | mith_pw | 66511 | 64506 | 0.970 | 12264 / 12421 | 12290 / 12442 | 3936 / 3982 | 3936 / 3982 |
| LIHC | DEG | 66511 | 14394 | 0.216 | 3708 / 12168 | 3826 / 12442 | 785 / 1845 | 785 / 1845 |
| LIHC | mith | 66511 | 66511 | 1 | 12421 / 12421 | 12442 / 12442 | 1845 / 1845 | 1845 / 1845 |
| LIHC | mith_LM | 66511 | 144 | 0.002 | 100 / 12421 | 100 / 12442 | 0 / 1845 | 0 / 1845 |
| LIHC | mith_pw | 66511 | 64716 | 0.973 | 12324 / 12421 | 12347 / 12442 | 1790 / 1845 | 1790 / 1845 |
| COAD | DEG | 66511 | 50 | 0.001 | 5 / 12168 | 33 / 12442 | 0 / 2750 | 0 / 2750 |
| COAD | mith | 66511 | 66511 | 1 | 12421 / 12421 | 12442 / 12442 | 2750 / 2750 | 2750 / 2750 |
| COAD | mith_LM | 66511 | 10 | 0.000 | 0 / 12421 | 4 / 12442 | 0 / 2750 | 0 / 2750 |
| COAD | mith_pw | 66511 | 63905 | 0.961 | 12285 / 12421 | 12308 / 12442 | 2672 / 2750 | 2672 / 2750 |

In [9]:
z = t[(t['srges'].str.startswith('0 /'))]
display(Markdown("Cancer x configuration pairs left with no drug on the validation line after the filter: " +
                 (', '.join(f'{c} {g}' for c, g in z.index) if len(z) else 'none') + ". "
                 "In COAD `mith_LM` the 10 kept instances give 4 drugs in `sRGES_all` (all lines) and none in `wsRGES`: they lie on lines without a weight. "
                 "The `mith` rows keep every instance because all their p-values are 0 (Table 6), and the `mith_pw` rows keep 96-97% for the same reason."))

Cancer x configuration pairs left with no drug on the validation line after the filter: LIHC mith_LM, COAD DEG, COAD mith_LM. In COAD `mith_LM` the 10 kept instances give 4 drugs in `sRGES_all` (all lines) and none in `wsRGES`: they lie on lines without a weight. The `mith` rows keep every instance because all their p-values are 0 (Table 6), and the `mith_pw` rows keep 96-97% for the same reason.

## 5. Offsets after the filter: rebuilt vs frozen

In [10]:
m = f64.merge(ff64, on=K, suffixes=('_b', '_f'))
assert len(m) == len(f64) == len(ff64)
rows = []
for (c, g, k), dd in m[m.collapse.isin(SR)].groupby(G, sort=False):
    rows.append({'cancer': c, 'config': g, 'collapse': k, 'n_drugs': len(dd),
                 'spearman': spearmanr(dd['score_b'], dd['score_f']).statistic if len(dd) >= 10 else np.nan,
                 'max_abs_diff': (dd['score_b'] - dd['score_f']).abs().max()})
r = pd.DataFrame(rows).pivot_table(index=['cancer', 'config'], columns='collapse', values=['n_drugs', 'spearman'])
r.columns = [f'{b} {a}' for a, b in r.columns]
r = r.reindex(pd.MultiIndex.from_product([CANCERS, CFG], names=['cancer', 'config']))
r = r[[f'{k} {v}' for k in SR for v in ('n_drugs', 'spearman')]]
display(Markdown("**Table 8.** Filtered scores with rebuilt (`bh05`) vs frozen (`bh05f`) offset tables, per cancer, configuration and "
                 "collapse: drugs scored after the filter (count; the same drugs in both variants) and Spearman of the two scores "
                 "(empty: fewer than 10 drugs or none). The two variants agree where the filter keeps almost every instance (`mith`, `mith_pw`) and "
                 "differ where it keeps few: there the rebuilt table is a mean over the pairs of the few kept instances "
                 "(COAD `DEG` 50, LIHC `mith_LM` 144 instances kept, Table 7)."))
md(r, index=True)

**Table 8.** Filtered scores with rebuilt (`bh05`) vs frozen (`bh05f`) offset tables, per cancer, configuration and collapse: drugs scored after the filter (count; the same drugs in both variants) and Spearman of the two scores (empty: fewer than 10 drugs or none). The two variants agree where the filter keeps almost every instance (`mith`, `mith_pw`) and differ where it keeps few: there the rebuilt table is a mean over the pairs of the few kept instances (COAD `DEG` 50, LIHC `mith_LM` 144 instances kept, Table 7).

| cancer | config | wsRGES n_drugs | wsRGES spearman | wsRGES_chen2017w n_drugs | wsRGES_chen2017w spearman | sRGES_all n_drugs | sRGES_all spearman | srges n_drugs | srges spearman |
|---|---|---|---|---|---|---|---|---|---|
| BRCA | DEG | 2257 | 0.920 | 2178 | 0.934 | 2467 | 0.901 | 1036 | 0.868 |
| BRCA | mith | 12417 | 1.000 | 12020 | 1.000 | 12438 | 1.000 | 3981 | 1.000 |
| BRCA | mith_LM | 1422 | 0.929 | 1331 | 0.926 | 1441 | 0.888 | 251 | 0.800 |
| BRCA | mith_pw | 12264 | 1.000 | 11857 | 1.000 | 12290 | 1.000 | 3936 | 1.000 |
| LIHC | DEG | 3708 | 0.972 | 3465 | 0.970 | 3826 | 0.957 | 785 | 0.998 |
| LIHC | mith | 12421 | 1 | 12024 | 1 | 12442 | 1 | 1845 | 1 |
| LIHC | mith_LM | 100 | 0.987 | 100 | 0.987 | 100 | 0.984 |  |  |
| LIHC | mith_pw | 12324 | 1.000 | 11926 | 1.000 | 12347 | 1.000 | 1790 | 1.000 |
| COAD | DEG | 5 |  | 5 |  | 33 | 0.917 |  |  |
| COAD | mith | 12421 | 1 | 12024 | 1 | 12442 | 1 | 2750 | 1.000 |
| COAD | mith_LM |  |  |  |  | 4 |  |  |  |
| COAD | mith_pw | 12285 | 1.000 | 11888 | 1.000 | 12308 | 1.000 | 2672 | 1.000 |

## 6. Outputs

- `other_outputs/binchen2017_validation/tcga_wsRGES_per_drug_DEG__binchen_r.tsv` (Chen 2017 reproduction: `DEG`, `binchen_r`, 58 lines).
- `tcga_wsRGES_per_drug_corrected.tsv` (corrected offsets, unfiltered; 58 lines for `DEG`, 64 for the MITHrIL configurations);
  `..._pvalue_filtered.tsv` (rebuilt offsets), `..._pvalue_filtered_frozen_offsets.tsv` (frozen offsets);
  `tcga_wsRGES_corrected_offsets_summary.tsv`.
- The same tables with the 58 DepMap weights in every configuration: `*_ccle58.tsv`.
- Code: `src/validations/chembl/wsRGES_tcga_corrected_offsets.py` (run from `src/` with `PYTHONPATH=.`).